# Lab 6 · Propagación de fallos y resiliencia de la red

Notebook para simular fallos aleatorios y ataques dirigidos sobre la red telco. Se reutilizan los CSV de red del Lab 5 incluidos en `data/`.

In [ ]:
# Instalación de dependencias básicas. En SageMaker normalmente ya están instaladas.
%pip -q install pandas numpy matplotlib networkx scikit-learn s3fs

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Por defecto se leen los CSV incluidos en la carpeta local data/.
# Si prefieres leer desde S3, cambia USE_S3=True y ajusta S3_PREFIX.
USE_S3 = False
DATA_DIR = Path('data')
S3_PREFIX = 's3://TU_BUCKET/TU_CARPETA'  # ejemplo: s3://mi-bucket/graph

def data_path(filename):
    return f"{S3_PREFIX}/{filename}" if USE_S3 else str(DATA_DIR / filename)

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

import random
import networkx as nx

## Parte A. Reconstruir el grafo

In [ ]:
nodes = pd.read_csv(data_path('network_nodes.csv'))
edges = pd.read_csv(data_path('network_edges.csv'))
cust = pd.read_csv(data_path('customer_nodes.csv'))

G = nx.Graph()
for _, r in nodes.iterrows():
    G.add_node(r['node_id'], node_type=r['node_type'], region=r['region'], zone=r['zone'])
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'])

cpn = cust.groupby('access_node_id').size()
for n in G.nodes():
    G.nodes[n]['n_clientes'] = int(cpn.get(n, 0))
TOTAL_CLIENTES = int(cust.shape[0])
print('Nodos:', G.number_of_nodes(), 'Enlaces:', G.number_of_edges(), 'Clientes:', TOTAL_CLIENTES)

In [ ]:
def clientes_conectados(H):
    cores = [n for n in H.nodes() if H.nodes[n].get('node_type') == 'core']
    if not cores:
        return 0
    conectados = 0
    for n, a in H.nodes(data=True):
        if a.get('n_clientes', 0) > 0:
            if any(nx.has_path(H, n, c) for c in cores):
                conectados += a['n_clientes']
    return int(conectados)

print('Clientes conectados al inicio:', clientes_conectados(G))

## Parte B. Simulación de fallos aleatorios

In [ ]:
def simular_fallos(G, orden, total_clientes):
    H = G.copy()
    serie_clientes, serie_componente, serie_componentes = [], [], []
    for nodo in orden:
        if nodo in H:
            H.remove_node(nodo)
        serie_clientes.append(clientes_conectados(H) / total_clientes)
        if H.number_of_nodes() > 0:
            mayor = max(nx.connected_components(H), key=len)
            serie_componente.append(len(mayor) / G.number_of_nodes())
            serie_componentes.append(nx.number_connected_components(H))
        else:
            serie_componente.append(0)
            serie_componentes.append(0)
    return np.array(serie_clientes), np.array(serie_componente), np.array(serie_componentes)

rep = 30
acc_cli = np.zeros(G.number_of_nodes())
acc_comp = np.zeros(G.number_of_nodes())
for _ in range(rep):
    orden = list(G.nodes())
    random.shuffle(orden)
    cli, comp, _ = simular_fallos(G, orden, TOTAL_CLIENTES)
    acc_cli += cli
    acc_comp += comp
aleatorio = acc_cli / rep
comp_aleatorio = acc_comp / rep

## Parte C. Ataque dirigido por centralidad

In [ ]:
bet = nx.betweenness_centrality(G)
orden_dirigido = sorted(G.nodes(), key=lambda n: bet[n], reverse=True)
dirigido, comp_dirigido, ncomp_dirigido = simular_fallos(G, orden_dirigido, TOTAL_CLIENTES)
print('Primeros nodos atacados:', orden_dirigido[:8])

In [ ]:
x = np.arange(1, G.number_of_nodes()+1) / G.number_of_nodes() * 100
plt.figure(figsize=(10,6))
plt.plot(x, aleatorio*100, label='Fallo aleatorio (media de 30)', lw=2)
plt.plot(x, dirigido*100, label='Ataque dirigido', lw=2)
plt.xlabel('% de nodos caídos')
plt.ylabel('% de clientes aún con servicio')
plt.title('Resiliencia: fallo aleatorio vs ataque dirigido')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab06_resiliencia_aleatorio_vs_dirigido.png', dpi=150, bbox_inches='tight')
plt.show()

## Parte D. Vulnerabilidades concretas

In [ ]:
H = G.copy()
umbral = None
for i, nodo in enumerate(orden_dirigido, start=1):
    H.remove_node(nodo)
    if H.number_of_nodes() == 0:
        break
    mayor = max(nx.connected_components(H), key=len)
    if len(mayor) < 0.5 * G.number_of_nodes():
        umbral = i
        print(f'Con {i} nodos dirigidos cae bajo el 50% de conectividad')
        break
if umbral is None:
    print('No cae bajo el 50% hasta retirar prácticamente toda la red.')

base = clientes_conectados(G)
impacto = {}
for n in G.nodes():
    H = G.copy()
    H.remove_node(n)
    impacto[n] = base - clientes_conectados(H)
impacto_df = pd.DataFrame({
    'node_id': list(impacto.keys()),
    'tipo': [G.nodes[n]['node_type'] for n in impacto.keys()],
    'region': [G.nodes[n]['region'] for n in impacto.keys()],
    'clientes_aislados': list(impacto.values()),
    'betweenness': [bet[n] for n in impacto.keys()]
}).sort_values('clientes_aislados', ascending=False)
display(impacto_df.head(12))
impacto_df.to_csv(OUTPUT_DIR/'lab06_impacto_individual.csv', index=False)

## Parte E. Simular medidas de refuerzo

In [ ]:
G2 = G.copy()
agg = [n for n in G2.nodes() if G2.nodes[n]['node_type'] == 'aggregation']
# Redundancia: anillo entre agregaciones
for i in range(len(agg)):
    G2.add_edge(agg[i], agg[(i+1) % len(agg)])
# Dos enlaces extra entre regiones con peor exposición
G2.add_edge('ACC-BIL-03', 'ACC-ZAR-03')
G2.add_edge('ACC-SEV-02', 'ACC-VAL-02')

bet2 = nx.betweenness_centrality(G2)
orden2 = sorted(G2.nodes(), key=lambda n: bet2[n], reverse=True)
dirigido2, _, _ = simular_fallos(G2, orden2, TOTAL_CLIENTES)

plt.figure(figsize=(10,6))
plt.plot(x, dirigido*100, label='Red original (ataque)', lw=2)
plt.plot(x, dirigido2*100, label='Red reforzada (ataque)', lw=2)
plt.xlabel('% de nodos caídos')
plt.ylabel('% de clientes con servicio')
plt.title('Efecto de añadir redundancia')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab06_refuerzo_redundancia.png', dpi=150, bbox_inches='tight')
plt.show()

## Cierre

Usa las figuras y la tabla `outputs/lab06_impacto_individual.csv` para redactar el entregable.